# P2 — House PTR data audit, cleaning log and provenance

**Source:** Frozen 2025 filing-index-year snapshot of the House PTR scraper output. **Unit:** one extracted transaction row. This notebook audits one source only. The filename contains Unity ID `sryan3`.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StrokeOfLuck/dsa405-part-2/blob/main/notebooks/DSA405_002_FA26_P2_sryan3.ipynb)

## 1. Setup and raw input

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

INPUT = Path("data/raw/house_ptr_transactions_filing_year_2025.csv.gz")
OUTPUT = Path("data/clean/house_ptr_2025_clean.csv")
if not INPUT.exists():
    # Running from notebooks/ inside a local checkout.
    parent_input = Path("..") / INPUT
    if parent_input.exists():
        INPUT, OUTPUT = parent_input, Path("..") / OUTPUT
    else:
        # Colab opens the notebook alone, without the other GitHub files.
        # Download this immutable snapshot into the temporary Colab runtime.
        try:
            import google.colab
        except ImportError as exc:
            raise FileNotFoundError(
                "Raw snapshot missing. Run from the repo root or open the GitHub notebook in Colab."
            ) from exc
        from urllib.request import urlretrieve
        INPUT.parent.mkdir(parents=True, exist_ok=True)
        url = (
            "https://raw.githubusercontent.com/StrokeOfLuck/dsa405-part-2/"
            "8c418bb26dccc4f3a66d67631fb07827126ad093/"
            "data/raw/house_ptr_transactions_filing_year_2025.csv.gz"
        )
        urlretrieve(url, INPUT)

# The frozen raw input must match the snapshot described in the README.
import gzip
import hashlib
with gzip.open(INPUT, "rb") as raw_stream:
    checksum = hashlib.sha256(raw_stream.read()).hexdigest()
assert checksum == "f71ebe00aa6b028ef57fe864158dc8501f67e3690adeffdd1090e1a9e21632a6", "Raw snapshot checksum mismatch"
raw = pd.read_csv(INPUT, dtype=str, keep_default_na=False)
print(f"Raw input: {len(raw):,} rows, {len(raw.columns)} columns")
print("Filing index years:", raw.source_year.value_counts(dropna=False).to_dict())
display(raw.head(3).iloc[:, :12])

## 2. Audit

Inspect every field retained for analysis. Empty strings are counted as missing because the input is deliberately loaded as text to preserve identifiers and original representations. The raw file remains unchanged.

In [ ]:
FIELDS = ["filing_id", "transaction_number_in_filing", "politician", "source_year", "owner", "asset_v8_2_cleaned", "transaction_type", "transaction_date", "amount_min", "amount_max", "amount_status", "needs_review", "original_pdf_url", "transaction_date_raw", "amount_raw"]
assert set(FIELDS).issubset(raw.columns)
EXPECTED = {"filing_id":"string", "transaction_number_in_filing":"integer", "politician":"string", "source_year":"integer", "owner":"string", "asset_v8_2_cleaned":"string", "transaction_type":"category", "transaction_date":"date", "amount_min":"dollars", "amount_max":"dollars", "amount_status":"category", "needs_review":"boolean", "original_pdf_url":"URL string", "transaction_date_raw":"string", "amount_raw":"string"}
audit = pd.DataFrame([{"field":k, "loaded dtype":str(raw[k].dtype), "intended type":EXPECTED[k], "missing count":int(raw[k].eq("").sum()), "missing rate":round(raw[k].eq("").mean(),4), "distinct incl. missing":int(raw[k].nunique(dropna=False))} for k in FIELDS])
display(audit)

In [ ]:
for col in ["source_year", "transaction_type", "amount_status", "needs_review"]:
    print(f"\n{col}: {raw[col].value_counts(dropna=False).to_dict()}")
for col in ["transaction_number_in_filing", "amount_min", "amount_max"]:
    vals = pd.to_numeric(raw[col].replace("",pd.NA), errors="coerce")
    print(col, "min:", vals.min(), "max:", vals.max(), "range:", vals.max()-vals.min(), "invalid nonempty:", int((vals.isna() & raw[col].ne("")).sum()))
for col in ["transaction_date", "transaction_date_raw"]:
    dates = pd.to_datetime(raw[col].replace("",pd.NA), errors="coerce", format="%Y-%m-%d" if col=="transaction_date" else "%m/%d/%Y")
    print(col, "min:", dates.min(), "max:", dates.max(), "unparsed nonempty:", int((dates.isna() & raw[col].ne("")).sum()))

In [ ]:
checks = {
    "exact duplicate rows": int(raw.duplicated().sum()),
    "duplicate filing ID + transaction number": int(raw.duplicated(["filing_id","transaction_number_in_filing"]).sum()),
    "surrounding spaces in selected fields": int(sum(raw[k].ne(raw[k].str.strip()).sum() for k in FIELDS)),
    "replacement characters (possible text corruption)": int(sum(raw[k].str.contains("\ufffd", regex=False).sum() for k in FIELDS)),
    "common sentinel tokens in selected fields": int(sum(raw[k].str.strip().str.lower().isin(["n/a","na","null","none","999"]).sum() for k in FIELDS)),
    "range with min greater than max": int((pd.to_numeric(raw.amount_min,errors="coerce") > pd.to_numeric(raw.amount_max,errors="coerce")).sum()),
    "review-flagged rows": int(raw.needs_review.eq("True").sum()),
}
display(pd.Series(checks,name="count").to_frame())
print("Identifier leading-zero examples:", {k:raw[k].str.match(r"^0[0-9]+$").sum() for k in ["filing_id","transaction_number_in_filing"]})
print("Possible subtotal labels:", {k:int(raw[k].str.contains(r"^\s*(?:total|subtotal)\s*$",case=False,regex=True).sum()) for k in ["politician","asset_v8_2_cleaned"]})
print("Unusual transaction types:", sorted(set(raw.transaction_type)-{"P","S","S (partial)","E"}))

### Raw-date extraction check

The source-like `transaction_date_raw` may contain text captured from neighboring PDF cells. A strict date parse failure here does **not** prove that the resolved `transaction_date` is wrong. Check representative cases against `original_pdf_url`, especially rows with `needs_review = False`. Keep the raw text for audit; do not silently trim it.

In [ ]:
raw_date_parsed = pd.to_datetime(raw.transaction_date_raw.replace("",pd.NA), errors="coerce", format="%m/%d/%Y")
raw_date_unparsed = raw.transaction_date_raw.ne("") & raw_date_parsed.isna()
print("Raw date text not a standalone MM/DD/YYYY date:",int(raw_date_unparsed.sum()))
print("Of these, not flagged by scraper:",int((raw_date_unparsed & raw.needs_review.eq("False")).sum()))
examples = raw.loc[raw_date_unparsed,["filing_id","transaction_number_in_filing","transaction_date_raw","transaction_date","needs_review","original_pdf_url"]].copy()
examples["transaction_date_raw"] = examples.transaction_date_raw.str.slice(0,90)
display(examples.head(8))

**Interpretation to finish after reviewing output:** Record which checked defect classes were absent. Inspect the review-flagged cases and any exceptions against `original_pdf_url`; distinguish an extraction problem from a legitimate disclosure. Do not silently guess corrections.

## 3. Data dictionary

In [ ]:
definitions = {
"filing_id":("identifier","House filing identifier; preserve as text"),
"transaction_number_in_filing":("count","Transaction row position within a filing; together with filing_id is a proposed key"),
"politician":("person","Name as supplied by scraper; not a stable person ID"),
"source_year":("calendar year","Disclosure index year, not necessarily transaction year"),
"owner":("disclosed owner","Blank may mean owner not separately specified"),
"asset_v8_2_cleaned":("asset name","Parser-resolved text; check linked PDF when uncertain"),
"transaction_type":("disclosure code","P purchase; S sale; E exchange; preserve partial-sale distinction"),
"transaction_date":("calendar date","Date reported for transaction; YYYY-MM-DD in parser output"),
"amount_min":("USD lower bound","Lower end of disclosed range, not exact amount"),
"amount_max":("USD upper bound","Upper end of disclosed range; may be absent for open-ended or exact amounts"),
"amount_status":("status code","Explains amount interpretation; inspect observed categories"),
"needs_review":("boolean","Scraper review flag; False is not independent proof of accuracy"),
"original_pdf_url":("URL","Link to source disclosure PDF"),
"transaction_date_raw":("text date","Parser's source-like date text retained for comparison"),
"amount_raw":("text amount","Parser's source-like amount text retained for comparison"),
}
valid_ranges = {"source_year":"2025 for this snapshot", "transaction_number_in_filing":"positive integer", "transaction_type":"P, S, S (partial), E; verify against source", "transaction_date":"valid calendar date", "amount_min":"nonnegative USD, when known", "amount_max":"nonnegative USD or missing if not bounded", "needs_review":"True or False"}
dictionary=pd.DataFrame([{"variable":k,"type":EXPECTED[k],"units":definitions[k][0],"factor levels":", ".join(sorted(raw[k].unique())) if raw[k].nunique()<=30 else "—", "valid range":valid_ranges.get(k,"Source-defined; inspect PDF"),"missing count":int(raw[k].eq("").sum()),"missing means":"Blank in extracted file; check source PDF before interpreting" if raw[k].eq("").any() else "None in snapshot", "notes":definitions[k][1]} for k in FIELDS])
display(dictionary)

## 4. Cleaning decisions and quantified log

The input is an already processed scraper export. Keep changes modest. Every entry below is a decision with a count, reason, and information-loss statement; the untouched raw snapshot permits reversal. The notebook does not alter ambiguous extracted values.

In [ ]:
clean = raw[FIELDS].copy()
log = []
def record(column, change, affected, why, lost):
    log.append({"#":len(log)+1,"column":column,"change":change,"rows/cells affected":int(affected),"why":why,"what is lost":lost,"how to reverse":"Re-read corresponding columns/rows from data/raw snapshot using filing_id and transaction_number_in_filing (or original row position)."})
record("other 46 columns", "Retained only 15 audit fields in clean output", len(raw), "A compact transaction-level analysis table is easier to inspect; full extraction diagnostics remain in raw file.", "Other 46 columns are omitted from clean output but preserved in raw snapshot.")
for col in ["politician","owner","asset_v8_2_cleaned","transaction_type","amount_status","original_pdf_url"]:
    stripped=clean[col].str.strip()
    count=int(clean[col].ne(stripped).sum())
    clean[col]=stripped
    if count: record(col,"Stripped surrounding whitespace",count,"Whitespace changes matching and category counts without representing a different disclosure.","Original surrounding spaces; raw field preserved in snapshot.")
for col in ["transaction_number_in_filing","source_year"]:
    parsed=pd.to_numeric(clean[col].replace("",pd.NA),errors="coerce").astype("Int64")
    bad=int(parsed.isna().sum())
    clean[col]=parsed
    record(col,"Parsed as nullable integer",len(clean)-bad,"These fields are integer indices rather than measured strings.","Original number formatting in clean file; unchanged text remains in raw snapshot.")
for col in ["amount_min","amount_max"]:
    parsed=pd.to_numeric(clean[col].replace("",pd.NA),errors="coerce")
    bad=int(parsed.isna().sum())
    clean[col]=parsed
    record(col,"Parsed as nullable dollar bound",len(clean)-bad,"Numeric bounds support validity checks; blanks remain missing.","Original number formatting in clean file; unchanged text remains in raw snapshot.")
parsed_date=pd.to_datetime(clean.transaction_date.replace("",pd.NA),errors="coerce",format="%Y-%m-%d")
record("transaction_date","Parsed ISO date",int(parsed_date.notna().sum()),"Date typing enables valid chronological comparisons.","Original date string format in clean file; original text is in raw snapshot and transaction_date_raw.")
clean.transaction_date=parsed_date
# Preserve rows, including any flagged for review. Do not convert blanks into asserted facts.
log_df=pd.DataFrame(log)
display(log_df)

**Review before submitting:** Check all converted blanks and any values that failed parsing. Add one log row per further decision, including an exact count and what would be lost. Do not list hypothetical operations as completed.

In [ ]:
for col in ["transaction_number_in_filing","source_year","amount_min","amount_max","transaction_date"]:
    source_nonempty=raw[col].ne("")
    unexpectedly_missing=clean[col].isna() & source_nonempty
    print(col, "nonempty inputs lost during parsing:", int(unexpectedly_missing.sum()))
    if unexpectedly_missing.any(): display(raw.loc[unexpectedly_missing,["filing_id","transaction_number_in_filing",col,"original_pdf_url"]].head(10))
print("Rows flagged for review (retained):", int(clean.needs_review.eq("True").sum()))

## 5. Row and column accounting

In [ ]:
accounting = pd.DataFrame([
("Raw rows",len(raw)),("Exact duplicate rows removed",0),("Near-duplicate rows removed",0),("Other rows removed",0),("Clean rows",len(clean)),
("Raw columns",raw.shape[1]),("Columns dropped",raw.shape[1]-clean.shape[1]),("Columns added",0),("Clean columns",clean.shape[1])],columns=["item","count"])
assert len(raw)==len(clean)
assert raw.shape[1]-(raw.shape[1]-clean.shape[1])==clean.shape[1]
assert not clean.duplicated(["filing_id","transaction_number_in_filing"]).any(), "Investigate duplicate proposed keys"
display(accounting)
OUTPUT.parent.mkdir(parents=True,exist_ok=True)
clean.to_csv(OUTPUT,index=False)
print("Wrote:",OUTPUT)

## 6. Provenance brief (review and revise; maximum 200 words)

This dataset contains transaction rows extracted from U.S. House periodic transaction disclosures listed in the House Clerk's 2025 disclosure index. Members file disclosures to report transactions; the House publishes the source PDFs. The independent House PTR scraper converted those PDFs into a transaction table and retained a link to each source document. This snapshot contains every row labeled `source_year = 2025` in one frozen scraper export. That label describes the filing index year, so some transactions may have happened in other years.

The table can describe what the scraper found in the disclosed filings, including reported dates, transaction types and value ranges. It cannot establish exact trade values, capture undisclosed transactions, or guarantee that PDF extraction interpreted every filing correctly. Flagged records and surprising values require checking the linked source PDF.

**Before submission:** Check this wording against the actual scraper pipeline and course expectations. Add any specific findings from the audit above.

## Submission check

- [ ] Review audit output, exception rows and sample PDFs.
- [ ] Confirm dictionary ranges and category meanings with the source.
- [ ] Add decisions made during manual review to the log with exact counts.
- [ ] Confirm raw/clean accounting and rerun from a restarted kernel.
- [ ] Attach self-scored rubric and submit notebook plus repo URL.